In [7]:
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.appName("CDC with Iceberg").getOrCreate()
print("1. Đã khởi động Spark")

spark.sql("CREATE NAMESPACE IF NOT EXISTS local_catalog.tiki")
print("2. Đã kết nối namespace 'tiki'")

# 3. Đọc đúng file dữ liệu của ngày hôm nay
today_str = datetime.now().strftime("%Y-%m-%d")
file_path = f"/home/jovyan/work/data/tiki_skincare_{today_str}.csv"

df_spark = spark.read.csv(file_path, header=True, inferSchema=True)

# 4. Đưa dữ liệu hôm nay vào "Phòng chờ" (Tạo bảng tạm trên RAM)
df_spark.createOrReplaceTempView("du_lieu_hom_nay")
print(f"3. Đã đọc file {file_path} lên phòng chờ")

# 5. THỰC HIỆN MA THUẬT MERGE INTO (UPSERT)
print("4. Đang đối chiếu và cập nhật dữ liệu vào Iceberg...")
spark.sql("""
    MERGE INTO local_catalog.tiki.products AS t
    USING du_lieu_hom_nay AS s
    ON t.id = s.id
    WHEN MATCHED THEN
        UPDATE SET 
            t.price = s.price,
            t.discount_rate = s.discount_rate,
            t.rating_average = s.rating_average,
            t.review_count = s.review_count,
            t.quantity_sold = s.quantity_sold
    WHEN NOT MATCHED THEN
        INSERT *
""")
print("-> Hoàn tất việc gộp dữ liệu (Merge)!")

# 6. Xem lại Top 10
print("\nTop 10 thương hiệu mới nhất: ")
spark.sql("""
    SELECT 
        brand_name, count(*) as so_luong_sp, round(avg(price), 0) as gia_trung_binh,
        round(avg(discount_rate), 2) as phan_tram_giam_trung_binh,
        round(avg(rating_average), 1) as danh_gia_trung_binh
    FROM local_catalog.tiki.products
    GROUP BY brand_name
    ORDER BY so_luong_sp DESC
    LIMIT 10
""").show(truncate=False)

1. Đã khởi động Spark
2. Đã kết nối namespace 'tiki'
3. Đã đọc file /home/jovyan/work/data/tiki_skincare_2026-05-12.csv lên phòng chờ
4. Đang đối chiếu và cập nhật dữ liệu vào Iceberg...
-> Hoàn tất việc gộp dữ liệu (Merge)!

Top 10 thương hiệu mới nhất: 
+---------------------------+-----------+--------------+-------------------------+-------------------+
|brand_name                 |so_luong_sp|gia_trung_binh|phan_tram_giam_trung_binh|danh_gia_trung_binh|
+---------------------------+-----------+--------------+-------------------------+-------------------+
|La Roche-Posay             |81         |476142.0      |0.43                     |2.6                |
|The Cocoon Original Vietnam|75         |248457.0      |8.07                     |2.8                |
|dermalogica                |48         |1336756.0     |0.0                      |0.3                |
|Hada Labo                  |43         |167128.0      |22.79                    |3.8                |
|Dabo                  